**Description** : Cette première étape est dédiée à la préparation des données brutes issues de LinkedIn.

**Objectifs** :  suppression des colonnes , récupération le nom de curent_comany et uniformisation des formats (villes, noms d'entreprises).

**Résultat** : Un fichier CSV propre prêt pour l'analyse de réseau.

**Importation les bibliothéques nécessaires**

In [1]:
import pandas as pd
import json
import numpy as np
import re

**Exploration des données**

In [2]:
#récupération le chemin d'accés des données
file_path = "../data/LinkedIn people profiles datasets.csv"

In [ ]:
# Lecture du fichier
df = pd.read_csv(file_path)

# Affichage des 5 premières lignes
df.head()

,timestamp,id,name,city,country_code,region,current_company:company_id,current_company:name,position,following,...,people_also_viewed,educations_details,education,avatar,languages,certifications,recommendations,recommendations_count,volunteer_experience,сourses
0,2023-01-10,catherinemcilkenny,"Catherine Fitzpatrick (McIlkenny), B.A",Canada,CA,NaN,NaN,NaN,Snr Business Analyst at Emploi et Développemen...,NaN,...,"[{""profile_link"":""https://ca.linkedin.com/in/l...",Queen's University Belfast,"[{""degree"":""Bachelor of Arts (B.A.) Honours"",""...",https://media.licdn.com/dms/image/C4E03AQEcz_j...,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-12-17,margot-bon-51a04624,Margot Bon,"The Randstad, Netherlands",NL,EU,gemeente-utrecht,Gemeente Utrecht,Communicatieadviseur Corporate & Strategie Gem...,NaN,...,"[{""profile_link"":""https://nl.linkedin.com/in/j...",NaN,"[{""degree"":""Scrum en Agile werken"",""end_year"":...",https://static.licdn.com/sc/h/244xhbkr7g40x6bs...,"[{""subtitle"":""Professional working proficiency...","[{""meta"":""Issued Jun 2013"",""subtitle"":""Van der...",Menno H. Poort “Ik werk al jaren prettig met M...,2.0,"[{""cause"":"""",""duration"":""Sep 2010 Jul 2020 9 y...",NaN
2,2023-05-17,mike-dean-8509a193,Mike Dean,"England, United Kingdom",UK,NaN,network-rail,Network Rail,Network Data Manager at Network Rail,NaN,...,"[{""profile_link"":""https://uk.linkedin.com/in/g...",Brighton Polytechnic,"[{""degree"":""2:2"",""end_year"":""1991"",""field"":""El...",https://media.licdn.com/dms/image/C4D03AQHLj-Z...,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-05-29,giovanna-panarella-99a0a4167,Giovanna Panarella,"Avellino, Campania, Italy",IT,EU,NaN,Freelance,Architetto (Freelance),500.0,...,"[{""profile_link"":""https://it.linkedin.com/in/e...",Università di Camerino,"[{""degree"":""“Corso di aggiornamento profession...",https://media-exp1.licdn.com/dms/image/C4D03AQ...,NaN,NaN,NaN,NaN,"[{""cause"":""Arts and Culture"",""duration"":""Jan 2...",NaN
4,2022-12-06,steve-latimer-3364327,Steve Latimer,"Ontario, Canada",CA,NaN,mid-range-computer-group-inc.,Mid-Range Computer Group Inc.,Senior Account Executive at Mid-Range Computer...,NaN,...,"[{""profile_link"":""https://ca.linkedin.com/in/d...",St. Michael's College School,"[{""degree"":"""",""end_year"":""1978"",""field"":"""",""me...",NaN,NaN,"[{""meta"":""Issued Jan 2022 See credential"",""sub...","Blake Reeves “If I was a customer, I would wan...",1.0,NaN,NaN


**nettoyage des données**

In [3]:
# 1. Définir la liste exacte des colonnes vérifiées
colonnes_a_garder = [
    "id", 
    "name", 
    "city", 
    "country_code", 
    "region", 
    
     
    "current_company", 
     
    "educations_details"
]

Ici, j'ai décidé seulement de garder les colonnes dont j'ai besoin l'id , le nom de l'employé , le country_code , region , le nom de l'entreprise actuel et l'éducation (université).Toutes, ces colonnes sont nécessaires pour faire les analyses plus tard.

In [4]:
df_link = pd.read_csv(file_path, usecols=colonnes_a_garder)

In [ ]:
# 3. Vérifier la nouvelle forme du dataset et les premières lignes
print(f"Dimensions du dataset : {df_link.shape}")
df_link.head()

Dimensions du dataset : (1000, 7)


,id,name,city,country_code,region,current_company,educations_details
0,catherinemcilkenny,"Catherine Fitzpatrick (McIlkenny), B.A",Canada,CA,NaN,"{""name"":""""}",Queen's University Belfast
1,margot-bon-51a04624,Margot Bon,"The Randstad, Netherlands",NL,EU,"{""company_id"":""gemeente-utrecht"",""industry"":""G...",NaN
2,mike-dean-8509a193,Mike Dean,"England, United Kingdom",UK,NaN,"{""company_id"":""network-rail"",""link"":""https://s...",Brighton Polytechnic
3,giovanna-panarella-99a0a4167,Giovanna Panarella,"Avellino, Campania, Italy",IT,EU,"{""link"":null,""name"":""Freelance""}",Università di Camerino
4,steve-latimer-3364327,Steve Latimer,"Ontario, Canada",CA,NaN,"{""company_id"":""mid-range-computer-group-inc."",...",St. Michael's College School


In [5]:
# 1. Création de la fonction modifiée pour extraire le nom et l'industrie
def extract_company_info(json_string):
    # Si la case est vide (NaN), on retourne deux valeurs nulles
    if pd.isna(json_string):
        return pd.Series([np.nan, np.nan])
    
    try:
        # On transforme le texte en dictionnaire Python
        data = json.loads(json_string)
        
        # On récupère les deux valeurs
        name = data.get("name", np.nan)
        industry = data.get("industry", np.nan)
        
        # On retourne une Série Pandas contenant les deux informations
        return pd.Series([name, industry])
        
    except (json.JSONDecodeError, TypeError):
        # Si le texte n'est pas un JSON valide, on retourne deux valeurs nulles
        return pd.Series([np.nan, np.nan])

Dans la colonne "current_company" est présenté sous fome de JSON . Donc à partir de ça , j'ai pu extraire le nom de company actuel et l'industrie (IT,Santé...)

In [14]:
# 2. Application de la fonction et création des DEUX colonnes en une seule ligne
df_link[['current_company_name', 'industry']] = df_link['current_company'].apply(extract_company_info)

# 3. Suppression de la colonne d'origine devenue inutile
df_link = df_link.drop(columns=['current_company'])

# 4. Affichage des résultats pour vérifier
df_link[['name', 'current_company_name', 'industry']].head(10)

,name,current_company_name,industry
0,"Catherine Fitzpatrick (McIlkenny), B.A",,NaN
1,Margot Bon,Gemeente Utrecht,Government Administration
2,Mike Dean,Network Rail,NaN
3,Giovanna Panarella,Freelance,NaN
4,Steve Latimer,Mid-Range Computer Group Inc.,IT Services and IT Consulting
5,Manuela Dias,Ericsson,IT Services and IT Consulting
6,Gerard Ludovic Wan,,NaN
7,Walter Patricio Rehbein Oyarzo,,NaN
8,Kyle Huddle,Clear Channel Airport,Marketing and Advertising
9,Marcus Singleton,,NaN


In [15]:
df_link.head(10)

,id,name,city,country_code,region,educations_details,current_company_name,industry
0,catherinemcilkenny,"Catherine Fitzpatrick (McIlkenny), B.A",Canada,CA,NaN,queens university belfast,,NaN
1,margot-bon-51a04624,Margot Bon,"The Randstad, Netherlands",NL,EU,NaN,Gemeente Utrecht,Government Administration
2,mike-dean-8509a193,Mike Dean,"England, United Kingdom",UK,NaN,brighton polytechnic,Network Rail,NaN
3,giovanna-panarella-99a0a4167,Giovanna Panarella,"Avellino, Campania, Italy",IT,EU,universit di camerino,Freelance,NaN
4,steve-latimer-3364327,Steve Latimer,"Ontario, Canada",CA,NaN,st michaels college school,Mid-Range Computer Group Inc.,IT Services and IT Consulting
5,manuela-dias-b868b29b,Manuela Dias,"Madrid, Community of Madrid, Spain",ES,EU,NaN,Ericsson,IT Services and IT Consulting
6,gerardludovicwan,Gerard Ludovic Wan,Canada,CA,NaN,mcgill university,,NaN
7,walter-patricio-rehbein-oyarzo-434613bb,Walter Patricio Rehbein Oyarzo,Chile,CL,SA,NaN,,NaN
8,kyle-huddle-a67a21b8,Kyle Huddle,"Allentown, Pennsylvania, United States",US,NaN,lehigh carbon community college,Clear Channel Airport,Marketing and Advertising
9,marcus-singleton-b779b414,Marcus Singleton,"Delémont, Jura, Switzerland",CH,EU,NaN,,NaN


In [16]:
#Afficher les différents types des colonnes
df_link.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    1000 non-null   str  
 1   name                  1000 non-null   str  
 2   city                  949 non-null    str  
 3   country_code          998 non-null    str  
 4   region                456 non-null    str  
 5   educations_details    682 non-null    str  
 6   current_company_name  973 non-null    str  
 7   industry              438 non-null    str  
dtypes: str(8)
memory usage: 62.6 KB


A partir de ça j'ai pu compris quels sont les caractérisques communs entre les employés (comme celle ci industry et education) .Mais , j'ai bien conclu qu'il y a pas beaucoup des entreprises communs.

In [17]:
# 1. Voir l'état actuel des valeurs manquantes (pour votre rapport)
print("Valeurs manquantes AVANT nettoyage :")
print(df_link.isnull().sum())
print("-" * 30)

Valeurs manquantes AVANT nettoyage :
id                        0
name                      0
city                     51
country_code              2
region                  544
educations_details      318
current_company_name     27
industry                562
dtype: int64
------------------------------


J'ai remarqué que j'ai des valeurs manquantes (donc l'utilisation des techniques de machine learning pour la détection des communautés sera un peu difficile )

In [18]:
# 1. Création de la fonction de nettoyage
def clean_text(text):
    # On ignore les valeurs vides ou notre texte par défaut
    if pd.isna(text) or text == "Non renseigné":
        return text
    
    # Convertir en chaîne de caractères
    text = str(text)
    
    # Mettre tout en minuscule
    text = text.lower()
    
    # Garder uniquement les lettres (a-z), les chiffres (0-9) et les espaces (\s)
    # Le ^ signifie "tout ce qui N'EST PAS" ça, on le remplace par rien ('')
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Supprimer les espaces multiples au milieu et les espaces aux extrémités
    text = ' '.join(text.split())
    
    return text

In [19]:
# 2. Application de la fonction aux trois colonnes
colonnes_a_nettoyer = ['educations_details', 'current_company_name', 'industry']

for col in colonnes_a_nettoyer:
    df_link[col] = df_link[col].apply(clean_text)

In [20]:
# 3. Affichage des valeurs les plus fréquentes (apparition >= 2)
print("--- RÉSULTATS DES VALEURS FRÉQUENTES ---\n")

for col in colonnes_a_nettoyer:
    print(f"🔹 Colonne : {col.upper()}")
    # On compte les occurrences de chaque valeur
    counts = df_link[col].value_counts()
    
    # On filtre pour ne garder que celles qui apparaissent au moins 2 fois
    # On exclut aussi "non renseigné" pour avoir de vraies statistiques
    frequent_values = counts[(counts >= 2) & (counts.index != "non renseigné")]
    
    if frequent_values.empty:
        print("Aucune valeur n'apparaît plus d'une fois.")
    else:
        # On affiche le top 15 pour éviter de surcharger l'écran
        print(frequent_values.head(15))
    print("\n" + "-"*40 + "\n")

--- RÉSULTATS DES VALEURS FRÉQUENTES ---

🔹 Colonne : EDUCATIONS_DETAILS
educations_details
                                             13
university of california los angeles          3
georgia state university                      3
rochester institute of technology             3
the university of texas at austin             3
deakin university                             3
queens university belfast                     2
mcgill university                             2
ecole marocaine des sciences de lingnieur     2
jamia millia islamia                          2
southern illinois university edwardsville     2
kent state university                         2
universidad nacional de colombia              2
quinnipiac university                         2
capilano university                           2
Name: count, dtype: int64

----------------------------------------

🔹 Colonne : CURRENT_COMPANY_NAME
current_company_name
                               147
self employed                 

In [21]:
# Chemin direct vers le dossier que vous avez créé
txt_filename = "../resultats/statistiques_frequence.txt"

with open(txt_filename, "w", encoding="utf-8") as f:
    f.write("=== ANALYSE DES VALEURS FRÉQUENTES (Apparition >= 2) ===\n\n")
    
    for col in ['educations_details', 'current_company_name', 'industry']:
        f.write(f"🔹 COLONNE : {col.upper()}\n")
        counts = df_link[col].value_counts()
        
        # On filtre les valeurs fréquentes et on enlève "non renseigné"
        frequent_values = counts[(counts >= 2) & (counts.index != "non renseigné")]
        
        if frequent_values.empty:
            f.write("Aucune valeur n'apparaît plus d'une fois.\n")
        else:
            # On écrit les résultats dans le fichier
            f.write(frequent_values.to_string())
        
        f.write("\n\n" + "-"*40 + "\n\n")

print(f"✅ Parfait ! Le fichier à montrer à votre enseignante a été sauvegardé dans : {txt_filename}")

✅ Parfait ! Le fichier à montrer à votre enseignante a été sauvegardé dans : ../resultats/statistiques_frequence.txt


In [22]:
# Sauvegarde du dataset propre dans le dossier data
chemin_csv_propre = "../data/linkedin_cleaned_data.csv"

df_link.to_csv(chemin_csv_propre, index=False, encoding='utf-8')

print(f"✅ Dataset propre sauvegardé avec succès : {chemin_csv_propre}")

✅ Dataset propre sauvegardé avec succès : ../data/linkedin_cleaned_data.csv
